# Global WSI native versus watershed reconciliation

This is a small, explicitly gated inspection notebook for the regenerated SLIDE-0330 all-channel half crop. The fork WSI method always infers unresolved tile planes and applies the selected resolver only after stitching. No standalone unresolved artifact or native TIFF export is created by this notebook.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path('/data1/lowes/ratnayn/Codex/projects/mIF-pipeline')
INSTANSEG_ROOT = REPO_ROOT.parent / 'instanseg'
CROP = Path('/data1/lowes/ratnayn/Codex/codex-scratch/mIF-pipeline/instanseg_watershed_production_smoke_all_channel_crop/SLIDE-0330/SLIDE-0330_all_channels_half_crop.ome.tif')
OUTPUT_ROOT = CROP.parent / 'global_resolver_comparison'
NATIVE_OUTPUT = OUTPUT_ROOT / 'SLIDE-0330_global_native.zarr'
WATERSHED_OUTPUT = OUTPUT_ROOT / 'SLIDE-0330_global_watershed.zarr'

# Change these only after the crop has been regenerated and sampled against its source.
RUN_NATIVE = False
RUN_WATERSHED = False
REUSE_COMPLETED = True
WSI_TILE_SIZE = 2048
WSI_OVERLAP = 80
WSI_DETECTION_SIZE = 20
WSI_BATCH_SIZE = 1
PIXEL_SIZE_UM = 0.325
MODEL_NAME = 'fluorescence_nuclei_and_cells'
SEGMENTATION_CHANNELS = [
    'R1_DAPI', 'R4_P19_POLYRAT', 'R4_GFP_POLY_AF488',
    'R6_CD45_CST_AF647', 'R6_PANCK_AE1_AE3_750',
    'R12_CD31_D8V9E_AF750', 'R7_NAK_ATPASE_555',
    'R8_F480_D2S9R_555', 'R9_CD68_E3O7V_488',
    'R12_CD3E_E4T1B_AF555',
]
REFERENCE_CHANNEL = 'R1_DAPI'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if str(INSTANSEG_ROOT) not in sys.path:
    sys.path.insert(0, str(INSTANSEG_ROOT))

In [ ]:
import tifffile
if not CROP.is_file():
    raise FileNotFoundError(f'Regenerate and validate the crop first: {CROP}')
try:
    fork_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=INSTANSEG_ROOT, text=True).strip()
except Exception as exc:
    fork_commit = f'git lookup failed: {exc}'
import instanseg
print({'instanseg_import': str(Path(instanseg.__file__).resolve()), 'fork_commit': fork_commit})
assert Path(instanseg.__file__).resolve().is_relative_to(INSTANSEG_ROOT), 'Restart with the InstanSeg fork on PYTHONPATH.'
with tifffile.TiffFile(CROP) as tif:
    series = tif.series[0]
    crop_shape = tuple(int(v) for v in series.shape)
    crop_axes = series.axes
    ome_xml = tif.ome_metadata or ''
    n_channels = crop_shape[0] if crop_axes == 'CYX' else len(tif.pages)
    physical_x = None
    try:
        import xml.etree.ElementTree as ET
        root = ET.fromstring(ome_xml)
        pixels = next(node for node in root.iter() if node.tag.endswith('Pixels'))
        physical_x = float(pixels.attrib.get('PhysicalSizeX'))
    except Exception:
        pass
print({'crop': str(CROP), 'axes': crop_axes, 'shape': crop_shape, 'channels': n_channels, 'physical_size_x_um': physical_x})
if crop_axes != 'CYX' or n_channels < 1:
    raise ValueError(f'Expected a CYX all-channel crop, got {crop_axes!r} {crop_shape!r}.')

In [ ]:
# Keep the selected order explicit and ensure the metadata-designated DAPI channel is the reference.
import re
channel_names = []
for match in re.finditer(r'<(?:[^:>]+:)?Channel\b[^>]*?Name=\"([^\"]*)\"', ome_xml):
    channel_names.append(match.group(1))
if len(channel_names) != n_channels:
    raise ValueError(f'OME metadata has {len(channel_names)} channel names for {n_channels} image channels.')
duplicates = sorted({name for name in channel_names if channel_names.count(name) > 1})
if duplicates:
    raise ValueError(f'Channel names must be unique: {duplicates}')
channel_to_index = {name: i for i, name in enumerate(channel_names)}
missing = [name for name in SEGMENTATION_CHANNELS if name not in channel_to_index]
if missing:
    raise KeyError(f'Missing production segmentation channels: {missing}')
CHANNEL_IDS = [channel_to_index[name] for name in SEGMENTATION_CHANNELS]
REFERENCE_CHANNEL_ID = channel_to_index[REFERENCE_CHANNEL]
assert CHANNEL_IDS[0] == REFERENCE_CHANNEL_ID
print({'channel_names_found': len(channel_names), 'reference_channel_id': REFERENCE_CHANNEL_ID, 'channel_ids': CHANNEL_IDS, 'channel_names_selected': SEGMENTATION_CHANNELS})

In [ ]:
from tiffslide import TiffSlide
import instanseg.inference_class as inference_class
inference_class.TiffSlide = TiffSlide
import zarr
_MODEL = None

def completed_output(path, method):
    if not path.is_dir():
        return False
    try:
        arr = zarr.open(str(path), mode='r')
        attrs = dict(arr.attrs)
        settings = attrs.get('wsi_settings', {})
        return (
            attrs.get('status') == 'complete'
            and attrs.get('resolution', {}).get('method') == method
            and Path(attrs.get('source_image', '')).resolve() == CROP.resolve()
            and list(attrs.get('channel_ids', [])) == CHANNEL_IDS
            and settings.get('tile_size') == WSI_TILE_SIZE
            and settings.get('overlap') == WSI_OVERLAP
            and settings.get('detection_size') == WSI_DETECTION_SIZE
            and settings.get('resolve_cell_and_nucleus') is True
        )
    except Exception:
        return False

def run_or_reuse(method, path, run):
    global _MODEL
    if completed_output(path, method) and REUSE_COMPLETED and not run:
        print(f'reusing validated {method}: {path}')
        return path
    if not run:
        print(f'{method} is gated off; set its RUN_* flag to True or provide a completed output: {path}')
        return None
    from instanseg import InstanSeg
    if _MODEL is None:
        _MODEL = InstanSeg(MODEL_NAME)
    return _MODEL.eval_whole_slide_image_global_normalization(
        str(CROP), channel_ids=CHANNEL_IDS, reference_channel_id=REFERENCE_CHANNEL_ID,
        pixel_size=PIXEL_SIZE_UM,
        tile_size=WSI_TILE_SIZE, overlap=WSI_OVERLAP, detection_size=WSI_DETECTION_SIZE,
        batch_size=WSI_BATCH_SIZE, output_path=path, overwrite=True,
        resolve_cell_and_nucleus=True, resolution_method=method,
    )

native_path = run_or_reuse('native', NATIVE_OUTPUT, RUN_NATIVE)
watershed_path = run_or_reuse('watershed', WATERSHED_OUTPUT, RUN_WATERSHED)

In [ ]:
def chunk_summary(path):
    if path is None:
        return None
    arr = zarr.open(str(path), mode='r')
    attrs = dict(arr.attrs)
    counts = [set(), set()]
    mismatched = 0
    for y0 in range(0, arr.shape[1], arr.chunks[1]):
        for x0 in range(0, arr.shape[2], arr.chunks[2]):
            y1, x1 = min(arr.shape[1], y0 + arr.chunks[1]), min(arr.shape[2], x0 + arr.chunks[2])
            n, c = np.asarray(arr[:, y0:y1, x0:x1])
            counts[0].update(int(v) for v in np.unique(n) if v > 0)
            counts[1].update(int(v) for v in np.unique(c) if v > 0)
            mismatched += int(np.count_nonzero((n > 0) & (n != c)))
    summary = {'path': str(path), 'shape': tuple(arr.shape), 'dtype': str(arr.dtype), 'status': attrs.get('status'), 'resolution': attrs.get('resolution'), 'validation': attrs.get('validation'), 'nuclear_ids': len(counts[0]), 'cell_ids': len(counts[1]), 'mismatched_nuclear_pixels': mismatched}
    validation = summary['validation'] or {}
    assert summary['shape'][0] == 2 and summary['dtype'] == 'int32' and summary['status'] == 'complete'
    assert mismatched == 0 and validation.get('nuclear_cell_ids_agree') is True
    assert validation.get('all_raw_nuclei_preserved') is True
    assert validation.get('one_final_cell_id_per_raw_nucleus') is True
    assert validation.get('all_proxy_cells_exact') is True
    return summary

native_summary = chunk_summary(native_path)
watershed_summary = chunk_summary(watershed_path)
display(native_summary)
display(watershed_summary)
if native_path is not None and watershed_path is not None:
    native = zarr.open(str(native_path), mode='r')
    watershed = zarr.open(str(watershed_path), mode='r')
    assert native.shape == watershed.shape
    nuclear_foreground_disagreement = 0
    nuclear_foreground_union = 0
    native_nuclear_pixels = 0
    watershed_nuclear_pixels = 0
    for y0 in range(0, native.shape[1], native.chunks[1]):
        for x0 in range(0, native.shape[2], native.chunks[2]):
            y1 = min(native.shape[1], y0 + native.chunks[1]); x1 = min(native.shape[2], x0 + native.chunks[2])
            n_native = np.asarray(native[0, y0:y1, x0:x1])
            n_watershed = np.asarray(watershed[0, y0:y1, x0:x1])
            native_fg = n_native > 0; watershed_fg = n_watershed > 0
            nuclear_foreground_disagreement += int(np.count_nonzero(native_fg != watershed_fg))
            nuclear_foreground_union += int(np.count_nonzero(native_fg | watershed_fg))
            native_nuclear_pixels += int(np.count_nonzero(native_fg))
            watershed_nuclear_pixels += int(np.count_nonzero(watershed_fg))
    cross_run_fraction = nuclear_foreground_disagreement / max(nuclear_foreground_union, 1)
    print({'native_vs_watershed_nuclear_foreground_disagreement': nuclear_foreground_disagreement, 'fraction_of_union': cross_run_fraction, 'native_nuclear_pixels': native_nuclear_pixels, 'watershed_nuclear_pixels': watershed_nuclear_pixels})
    if nuclear_foreground_disagreement:
        print('Note: these outputs came from separate GPU inference passes. Exact cross-run pixel equality is diagnostic, not a resolver invariant; each output is validated strictly against its own unresolved inference.')

In [ ]:
# Lazy whole-crop foreground overview using bounded strided Zarr reads.
if native_path is not None or watershed_path is not None:
    selected = native_path or watershed_path
    labels = zarr.open(str(selected), mode='r')
    factor = max(1, int(max(labels.shape[-2:]) / 1200))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    for ax, plane, title in zip(axes, (0, 1), ('nuclei foreground', 'cell foreground')):
        overview = np.asarray(labels[plane, ::factor, ::factor]) > 0
        ax.imshow(overview, cmap='gray', interpolation='nearest')
        ax.set_title(f'{title} (stride {factor})'); ax.axis('off')
    plt.show()

In [ ]:
# Native-resolution RGB fields with separate nucleus and cell outlines.
from skimage.segmentation import find_boundaries

paths = [('Global native', native_path), ('Global watershed', watershed_path)]
paths = [(name, path) for name, path in paths if path is not None]
DISPLAY_CHANNELS = ['R9_CD68_E3O7V_488', 'R12_CD3E_E4T1B_AF555', 'R1_DAPI']
DISPLAY_CHANNEL_IDS = [channel_to_index[name] for name in DISPLAY_CHANNELS]

def robust01(image, percentiles=(1.0, 99.8)):
    image = np.asarray(image, dtype=np.float32)
    low, high = np.percentile(image, percentiles)
    if high <= low:
        return np.zeros_like(image, dtype=np.float32)
    return np.clip((image - low) / (high - low), 0, 1)

def labels_at_native_grid(label_store, x, y, size, src_w, src_h):
    native_y = np.arange(y, y + size, dtype=np.int64)
    native_x = np.arange(x, x + size, dtype=np.int64)
    model_y = np.minimum(native_y * int(label_store.shape[1]) // src_h, int(label_store.shape[1]) - 1)
    model_x = np.minimum(native_x * int(label_store.shape[2]) // src_w, int(label_store.shape[2]) - 1)
    my0, my1 = int(model_y.min()), int(model_y.max()) + 1
    mx0, mx1 = int(model_x.min()), int(model_x.max()) + 1
    native_labels = np.empty((2, size, size), dtype=np.int32)
    for plane in range(2):
        block = np.asarray(label_store[plane, my0:my1, mx0:mx1])
        native_labels[plane] = np.take(np.take(block, model_y - my0, axis=0), model_x - mx0, axis=1)
    return native_labels

def show_outlines(axis, rgb, labels, mode, title):
    axis.imshow(rgb, interpolation='nearest')
    if mode in ('both', 'nuclei'):
        axis.contour(find_boundaries(labels[0], mode='outer'), levels=[0.5], colors=['cyan'], linewidths=0.7)
    if mode in ('both', 'cells'):
        axis.contour(find_boundaries(labels[1], mode='outer'), levels=[0.5], colors=['yellow'], linewidths=0.7)
    axis.set_title(title, fontsize=10)
    axis.axis('off')

if paths:
    src_h, src_w = crop_shape[-2:]
    panel_size = min(768, src_w, src_h)
    fields = [
        ('upper-left', src_w // 4, src_h // 4),
        ('center', (src_w - panel_size) // 2, (src_h - panel_size) // 2),
        ('lower-right', src_w - panel_size, src_h - panel_size),
    ]
    with tifffile.TiffFile(str(CROP)) as handle:
        input_store = handle.series[0].aszarr(level=0)
        try:
            input_array = zarr.open(input_store, mode='r')
            for field_name, x, y in fields:
                input_zoom = np.asarray(input_array.oindex[
                    DISPLAY_CHANNEL_IDS, slice(y, y + panel_size), slice(x, x + panel_size)
                ])
                rgb = np.stack([robust01(channel) for channel in input_zoom], axis=-1)
                resolved = {name: labels_at_native_grid(zarr.open(str(path), mode='r'), x, y, panel_size, src_w, src_h) for name, path in paths}
                fig, axes = plt.subplots(3, 1 + len(paths), figsize=(5 * (1 + len(paths)), 15), squeeze=False)
                for row, mode in enumerate(('both', 'nuclei', 'cells')):
                    axes[row, 0].imshow(rgb, interpolation='nearest')
                    axes[row, 0].set_title('Input RGB', fontsize=10)
                    axes[row, 0].set_ylabel(mode.title(), fontsize=11)
                    axes[row, 0].axis('off')
                    for column, (name, _) in enumerate(paths, start=1):
                        show_outlines(axes[row, column], rgb, resolved[name], mode, name)
                fig.suptitle(f'{field_name}: crop x={x}, y={y}, size={panel_size}px | RGB={DISPLAY_CHANNELS} | cyan=nuclei, yellow=cells', fontsize=12)
                fig.tight_layout(rect=(0, 0, 1, 0.97))
                plt.show()
        finally:
            input_store.close()